In [23]:
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv; load_dotenv()

from humaidclf import run_experiment
from humaidclf import build_token_index               # from budget.py
from humaidclf.batch import use_api_key_env           # context manager for key switching
from rules import RULES_1

# --- config ---
BASE = Path("Dataset/HumAID")
SPLITS = ["train"]             # or ["train","dev","test"]
MODEL = "gpt-4o-mini"
RULES = RULES_1
TAG = "modeS-RULES1"
DRYRUN_N = 20
POLL_SECS = 300
DO_ANALYSIS = True
OUT_ROOT = "runs"

BATCH_TOKEN_LIMIT = 2_000_000  # Tier-1 cap
SAFETY_MARGIN = 0.90           # 10% headroom
MAX_OUTPUT_TOKENS = 40

In [24]:
# --- discover datasets ---
def discover_tsvs(base: Path, splits: list[str]):
    items = []
    for event_dir in sorted([p for p in base.iterdir() if p.is_dir()]):
        event = event_dir.name
        for split in splits:
            tsv = event_dir / f"{event}_{split}.tsv"
            if tsv.exists():
                items.append({"event": event, "split": split, "tsv": str(tsv)})
    return pd.DataFrame(items)

df_sources = discover_tsvs(BASE, SPLITS)

df_sources

,event,split,tsv
0,california_wildfires_2018,train,Dataset\HumAID\california_wildfires_2018\calif...
1,canada_wildfires_2016,train,Dataset\HumAID\canada_wildfires_2016\canada_wi...
2,cyclone_idai_2019,train,Dataset\HumAID\cyclone_idai_2019\cyclone_idai_...
3,hurricane_dorian_2019,train,Dataset\HumAID\hurricane_dorian_2019\hurricane...
4,hurricane_florence_2018,train,Dataset\HumAID\hurricane_florence_2018\hurrica...
5,hurricane_harvey_2017,train,Dataset\HumAID\hurricane_harvey_2017\hurricane...
6,hurricane_irma_2017,train,Dataset\HumAID\hurricane_irma_2017\hurricane_i...
7,hurricane_maria_2017,train,Dataset\HumAID\hurricane_maria_2017\hurricane_...
8,kaikoura_earthquake_2016,train,Dataset\HumAID\kaikoura_earthquake_2016\kaikou...
9,kerala_floods_2018,train,Dataset\HumAID\kerala_floods_2018\kerala_flood...


# kerala_floods_2018

In [13]:
tsv_path = df_1.loc[df_1["event"].eq("kerala_floods_2018"), "tsv"].iat[0]
df_kerala_floods_2018_tsv = pd.read_csv(tsv_path, sep="\t")   # or: pd.read_table(tsv_path)
df_kerala_floods_2018_tsv.head()

,tweet_id,tweet_text,class_label
0,1031218893908406273,"Kerala Floods: More than 38,000 people rescued...",rescue_volunteering_or_donation_effort
1,1030767523342499842,@BDUTT While PayTm owner Shekhar donated Rs Te...,rescue_volunteering_or_donation_effort
2,1030385981973618688,#KeralaFloods Malayala Manorama makes epaper f...,rescue_volunteering_or_donation_effort
3,1031078567147319297,#KeralaSOS #KeralaFloods Lets centralise all r...,sympathy_and_support
4,1033187418889834497,@narendramodi @AmitShah PS @ kerala is doing t...,rescue_volunteering_or_donation_effort


In [15]:
# counts only
df_kerala_floods_2018_tsv["class_label"].value_counts()


class_label
rescue_volunteering_or_donation_effort    3005
other_relevant_information                 669
sympathy_and_support                       585
requests_or_urgent_needs                   413
not_humanitarian                           319
injured_or_dead_people                     254
infrastructure_and_utility_damage          207
caution_and_advice                          97
displaced_people_and_evacuations            39
Name: count, dtype: int64

In [1]:
from pathlib import Path
import pandas as pd
from typing import List
from dotenv import load_dotenv; load_dotenv()

from humaidclf import run_experiment
from humaidclf import build_token_index
from humaidclf.batch import use_api_key_env
from rules import RULES_1

# --- config ---
BASE = Path("Dataset/HumAID")
SPLITS: List[str] = ["train"]  # or ["train","dev","test"]

# --- discover datasets ---
def discover_tsvs(base: Path, splits: List[str]) -> pd.DataFrame:
    if not base.exists():
        print(f"[WARN] Base path not found: {base.resolve()}")
        return pd.DataFrame(columns=["event", "split", "tsv"])
    items = []
    for event_dir in sorted([p for p in base.iterdir() if p.is_dir()]):
        event = event_dir.name
        for split in splits:
            tsv = event_dir / f"{event}_{split}.tsv"
            if tsv.exists():
                items.append({"event": event, "split": split, "tsv": str(tsv)})
            else:
                print(f"[WARN] Missing file: {tsv}")
    return pd.DataFrame(items)

# --- helpers to run a list of datasets ---
def show_class_distribution(dflist: pd.DataFrame, label_col: str = "class_label"):
    if dflist.empty:
        print("[INFO] No datasets discovered.")
        return
    for _, row in dflist.iterrows():
        event, split, tsv = row["event"], row["split"], row["tsv"]
        print(f"\n=== Showing {event}/{split} ===")
        try:
            df_current_even = pd.read_csv(tsv, sep="\t")

            # Clean labels (drop NaN, trim whitespace) for class counting
            labels_clean = (
                df_current_even[label_col]
                .dropna()
                .astype(str)
                .str.strip()
            )

            # Print distribution (using cleaned labels)
            counts = labels_clean.value_counts()
            print(counts.to_string())

            # Print total number of classes
            n_classes = labels_clean.nunique()
            print(f"Total classes: {n_classes}")

        except Exception as e:
            print(f"[ERROR] {event}/{split}: {e}")

df_sources = discover_tsvs(BASE, SPLITS)
show_class_distribution(df_sources)



=== Showing california_wildfires_2018/train ===
class_label
injured_or_dead_people                    1362
rescue_volunteering_or_donation_effort     991
not_humanitarian                           923
other_relevant_information                 727
sympathy_and_support                       330
infrastructure_and_utility_damage          295
displaced_people_and_evacuations           258
missing_or_found_people                    125
caution_and_advice                          97
requests_or_urgent_needs                    55
Total classes: 10

=== Showing canada_wildfires_2016/train ===
class_label
rescue_volunteering_or_donation_effort    653
displaced_people_and_evacuations          266
other_relevant_information                218
infrastructure_and_utility_damage         176
sympathy_and_support                      113
caution_and_advice                         74
not_humanitarian                           55
requests_or_urgent_needs                   14
Total classes: 8

=== Show

In [2]:
# --- config ---
BASE_TEST = Path("Dataset/HumAID")
SPLITS_TEST: List[str] = ["test"]  # or ["train","dev","test"]

df_sources_test = discover_tsvs(BASE_TEST, SPLITS_TEST)
show_class_distribution(df_sources_test)



=== Showing california_wildfires_2018/test ===
class_label
injured_or_dead_people                    385
rescue_volunteering_or_donation_effort    280
not_humanitarian                          261
other_relevant_information                205
sympathy_and_support                       94
infrastructure_and_utility_damage          84
displaced_people_and_evacuations           72
missing_or_found_people                    36
caution_and_advice                         28
requests_or_urgent_needs                   16
Total classes: 10

=== Showing canada_wildfires_2016/test ===
class_label
rescue_volunteering_or_donation_effort    186
displaced_people_and_evacuations           75
other_relevant_information                 61
infrastructure_and_utility_damage          50
sympathy_and_support                       32
caution_and_advice                         21
not_humanitarian                           16
requests_or_urgent_needs                    4
Total classes: 8

=== Showing cyclone_

In [2]:
import pandas as pd

df = pd.read_csv("results/hurricane_dorian_2019/test/gpt-4o-mini/20251101-154343-modeS-RULES1-filtered-TIER1/predictions.csv")

def clean(s):
    return (s.astype(str)
             .str.strip()
             .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA}))

truth = clean(df["class_label"])
pred  = clean(df["predicted_label"])

truth_set = set(truth.dropna().unique())

mask_missing = pred.isna()                        # empty / missing predictions
mask_oos     = ~mask_missing & (~pred.isin(truth_set))

print("truth_set (size):", len(truth_set), truth_set)
print("Total rows:", len(df))
print("Missing preds:", int(mask_missing.sum()))
print("OOS preds    :", int(mask_oos.sum()))

print("\nOOS breakdown (predicted_label -> count):")
print(pred[mask_oos].value_counts())

print("\nMissing pred rows (first 5):")
print(df.loc[mask_missing, ["tweet_id","tweet_text"]].head())


truth_set (size): 9 {'other_relevant_information', 'caution_and_advice', 'displaced_people_and_evacuations', 'requests_or_urgent_needs', 'sympathy_and_support', 'rescue_volunteering_or_donation_effort', 'infrastructure_and_utility_damage', 'not_humanitarian', 'injured_or_dead_people'}
Total rows: 1508
Missing preds: 15
OOS preds    : 0

OOS breakdown (predicted_label -> count):
Series([], Name: count, dtype: int64)

Missing pred rows (first 5):
                tweet_id                                         tweet_text
86   1168189066505965568  This is a catastrophic storm with a death toll...
316  1168263285507907585  20% of all funds from my etsy for the next cou...
343  1168195273505591297  Hopefully everyone how should have evacuated l...
587  1168255248164671490  Another one. David Simon is so consumed with h...
854  1167785614332194816  I attribute the delay to God trying to wipe ou...


# Python script you can run to sanity-check Structured Outputs end-to-end (no project imports needed). It:

- Builds a tiny dummy dataframe (same columns you use).
- Calls /v1/chat/completions with a strict JSON schema.
- Handles both .message.parsed and JSON-in-content variants.
- Avoids params that gpt-5-mini dislikes (via _safe_body).
- Prints/returns a dataframe with predicted_label (and optional confidence).

In [26]:
# quick_so_probe.py  (fixed for Responses API)
import os, json, requests, re
import pandas as pd
from dotenv import load_dotenv; load_dotenv()

OPENAI_BASE = "https://api.openai.com/v1"
API_KEY = os.getenv("OPENAI_API_KEY_1") or os.getenv("OPENAI_API_KEY")
if not API_KEY:
    raise SystemExit("Set OPENAI_API_KEY_1 (or OPENAI_API_KEY).")
H_JSON = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}

def _schema(labels: list[str], keep_conf: bool) -> dict:
    props = {"label": {"type": "string", "enum": labels}}
    if keep_conf:
        props["confidence"] = {"type": "number", "minimum": 0, "maximum": 1}
    return {"type": "object", "properties": props, "required": ["label"], "additionalProperties": False}

def _normalize_label(text: str, labels: list[str]) -> str | None:
    if not text: return None
    t = text.strip().strip('"\'')

    for L in labels:
        if t.lower() == L.lower():
            return L

    m = re.search(r'"label"\s*:\s*"([^"]+)"', t)
    if m:
        cand = m.group(1).strip()
        for L in labels:
            if cand.lower() == L.lower():
                return L

    m = re.search(r'\blabel\s*:\s*([A-Za-z\-]+)', t)
    if m:
        cand = m.group(1).strip()
        for L in labels:
            if cand.lower() == L.lower():
                return L

    for L in labels:
        if L.lower() in t.lower():
            return L
    return None

def _extract_from_chat(resp_json, labels):
    msg = resp_json["choices"][0]["message"]
    parsed = msg.get("parsed")
    if isinstance(parsed, dict) and "label" in parsed:
        return str(parsed["label"]).strip(), parsed
    content = msg.get("content", "")
    if isinstance(content, list):
        text = " ".join([p.get("text","") for p in content if isinstance(p, dict)])
    else:
        text = str(content)
    try:
        obj = json.loads(text)
        if isinstance(obj, dict) and "label" in obj:
            return str(obj["label"]).strip(), obj
    except Exception:
        pass
    lab = _normalize_label(text, labels)
    return (lab, {"label": lab} if lab else {})

def _extract_from_responses(resp_json, labels):
    # Prefer parsed output (Structured Outputs on Responses API)
    if "output_parsed" in resp_json and isinstance(resp_json["output_parsed"], list) and resp_json["output_parsed"]:
        obj = resp_json["output_parsed"][0]
        if isinstance(obj, dict) and "label" in obj:
            return str(obj["label"]).strip(), obj

    # Fallback: Find the message type output (not reasoning)
    text = ""
    try:
        output_list = resp_json.get("output", [])
        for item in output_list:
            if item.get("type") == "message":
                text = item["content"][0]["text"]
                break
        # If no message type found, try the first output
        if not text and output_list:
            text = output_list[0]["content"][0]["text"]
    except Exception:
        pass
    
    try:
        obj = json.loads(text)
        if isinstance(obj, dict) and "label" in obj:
            return str(obj["label"]).strip(), obj
    except Exception:
        pass
    lab = _normalize_label(text, labels)
    return (lab, {"label": lab} if lab else {})

def quick_so_probe(model: str = "gpt-5-mini", rules: str = "") -> pd.DataFrame:
    df = pd.DataFrame([
        {"tweet_id": "t1", "tweet_text": "We need water and food urgently.",        "class_label": "Request-Help"},
        {"tweet_id": "t2", "tweet_text": "Roads are flooded near the river.",       "class_label": "Report-Event"},
        {"tweet_id": "t3", "tweet_text": "Thank you to the volunteers. Stay safe!", "class_label": "Other"},
    ])
    labels = ["Request-Help", "Report-Event", "Other"]
    is_gpt5 = model.lower().startswith("gpt-5")

    # gpt-5-mini is picky: keep schema minimal (no confidence); use Responses API
    schema = _schema(labels, keep_conf=not is_gpt5)
    response_format = {"type": "json_schema", "json_schema": {"name": "tweet_label", "schema": schema}}

    rows = []
    dumped = False
    for _, r in df.iterrows():
        user_prompt = (
            f"Classify the tweet into exactly one of: {', '.join(labels)}.\n"
            f"{rules or 'Return a JSON object that follows the provided schema.'}\n"
            f"Tweet: {r['tweet_text']}"
        )

        if is_gpt5:
            # --- Responses API path ---
            # GPT-5 models use reasoning which consumes tokens, so we need much higher limits
            body = {
                "model": model,
                "input": [{"role": "system", "content": "Follow the schema exactly."},
                          {"role": "user", "content": user_prompt}],
                "text": {
                    "format": {
                        "type": "json_schema",
                        "name": "tweet_label",
                        "schema": schema,
                        "strict": True
                    }
                },
                "max_output_tokens": 500,  # Increased from 40 to account for reasoning tokens
            }
            resp = requests.post(f"{OPENAI_BASE}/responses", headers=H_JSON, json=body, timeout=60)
        else:
            # --- Chat Completions path ---
            body = {
                "model": model,
                "messages": [{"role": "system", "content": "Follow the schema exactly."},
                             {"role": "user", "content": user_prompt}],
                "response_format": response_format,
                "max_tokens": 40,
                "temperature": 0.0,
                "top_p": 1,
            }
            resp = requests.post(f"{OPENAI_BASE}/chat/completions", headers=H_JSON, json=body, timeout=60)

        if resp.status_code != 200:
            print(">>> API error body:", resp.text[:600])
            resp.raise_for_status()

        j = resp.json()
        
        # Handle incomplete responses for Responses API (GPT-5)
        if is_gpt5 and j.get("status") == "incomplete":
            response_id = j.get("id")
            print(f"⏳ Response incomplete, polling for completion (ID: {response_id})...")
            import time
            max_retries = 10
            for retry in range(max_retries):
                time.sleep(2)  # Wait 2 seconds before polling
                poll_resp = requests.get(
                    f"{OPENAI_BASE}/responses/{response_id}",
                    headers={"Authorization": f"Bearer {API_KEY}"},
                    timeout=60
                )
                if poll_resp.status_code == 200:
                    j = poll_resp.json()
                    if j.get("status") == "completed":
                        print(f"✓ Response completed after {retry + 1} poll(s)")
                        break
                    elif j.get("status") == "failed":
                        print(f"✗ Response failed: {j.get('error')}")
                        break
            else:
                print(f"⚠ Response still incomplete after {max_retries} retries")
        
        if not dumped:
            print("\n--- RAW first response ---")
            print(json.dumps(j, indent=2)[:1200])
            print("----------------------------------\n")
            dumped = True

        if is_gpt5:
            label, parsed = _extract_from_responses(j, labels)
        else:
            label, parsed = _extract_from_chat(j, labels)

        rows.append({
            "tweet_id": r["tweet_id"],
            "tweet_text": r["tweet_text"],
            "class_label": r["class_label"],
            "predicted_label": label or "",
            "confidence": (parsed.get("confidence") if isinstance(parsed, dict) else None),
        })

    out = pd.DataFrame(rows)
    print(out)
    try:
        from sklearn.metrics import f1_score
        print("Macro-F1:", f1_score(out["class_label"], out["predicted_label"], average="macro"))
    except Exception:
        pass
    return out

In [27]:
# Try models you've been using; start with 4o-mini or gpt-5-mini
quick_so_probe(model="gpt-5-mini")


--- RAW first response ---
{
  "id": "resp_0bd64b23f0d3466600690d7cd893e08195ac528775fe4fb208",
  "object": "response",
  "created_at": 1762491608,
  "status": "completed",
  "background": false,
  "billing": {
    "payer": "developer"
  },
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "max_output_tokens": 500,
  "max_tool_calls": null,
  "model": "gpt-5-mini-2025-08-07",
  "output": [
    {
      "id": "rs_0bd64b23f0d3466600690d7cda15fc819586877ef4de8ea10b",
      "type": "reasoning",
      "summary": []
    },
    {
      "id": "msg_0bd64b23f0d3466600690d7cdb6dd881959005a34d0134a5b9",
      "type": "message",
      "status": "completed",
      "content": [
        {
          "type": "output_text",
          "annotations": [],
          "logprobs": [],
          "text": "{\"label\":\"Request-Help\"}"
        }
      ],
      "role": "assistant"
    }
  ],
  "parallel_tool_calls": true,
  "previous_response_id": null,
  "prompt_cache_key": null,
  "prompt_c

,tweet_id,tweet_text,class_label,predicted_label,confidence
0,t1,We need water and food urgently.,Request-Help,Request-Help,None
1,t2,Roads are flooded near the river.,Report-Event,Report-Event,None
2,t3,Thank you to the volunteers. Stay safe!,Other,Other,None


# GPT code for gpt-5 inference

In [28]:
# probe_humaid_like.py
import os, json, re, requests
import pandas as pd
from typing import List, Tuple, Dict, Any, Optional
from dotenv import load_dotenv; load_dotenv()

OPENAI_BASE = "https://api.openai.com/v1"
API_KEY = os.getenv("OPENAI_API_KEY_1") or os.getenv("OPENAI_API_KEY")
if not API_KEY:
    raise SystemExit("Set OPENAI_API_KEY_1 (or OPENAI_API_KEY).")
H_JSON = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}

# ----------------------------- helpers ---------------------------------

def _schema(labels: List[str], keep_conf: bool) -> Dict[str, Any]:
    props = {"label": {"type": "string", "enum": labels}}
    if keep_conf:
        props["confidence"] = {"type": "number", "minimum": 0, "maximum": 1}
    return {"type": "object", "properties": props, "required": ["label"], "additionalProperties": False}

def _normalize_label(text: str, labels: List[str]) -> Optional[str]:
    if not text:
        return None
    t = text.strip().strip('"\'')

    # exact case-insensitive match
    for L in labels:
        if t.lower() == L.lower():
            return L

    # JSON-ish: "label":"<X>"
    m = re.search(r'"label"\s*:\s*"([^"]+)"', t)
    if m:
        cand = m.group(1).strip()
        for L in labels:
            if cand.lower() == L.lower():
                return L

    # looser "label: X"
    m = re.search(r'\blabel\s*:\s*([A-Za-z0-9\-\_ ]+)', t)
    if m:
        cand = m.group(1).strip()
        for L in labels:
            if cand.lower() == L.lower():
                return L

    # last-resort substring
    for L in labels:
        if L.lower() in t.lower():
            return L
    return None

def _extract_from_chat(resp_json: Dict[str, Any], labels: List[str]) -> Tuple[Optional[str], Dict[str, Any]]:
    msg = resp_json["choices"][0]["message"]
    parsed = msg.get("parsed")
    if isinstance(parsed, dict) and "label" in parsed:
        return str(parsed["label"]).strip(), parsed

    content = msg.get("content", "")
    if isinstance(content, list):
        text = " ".join([p.get("text", "") for p in content if isinstance(p, dict)])
    else:
        text = str(content)

    try:
        obj = json.loads(text)
        if isinstance(obj, dict) and "label" in obj:
            return str(obj["label"]).strip(), obj
    except Exception:
        pass

    lab = _normalize_label(text, labels)
    return lab, ({"label": lab} if lab else {})

def _extract_from_responses(resp_json: Dict[str, Any], labels: List[str]) -> Tuple[Optional[str], Dict[str, Any]]:
    # Preferred: Structured Outputs parsed payload (array)
    op = resp_json.get("output_parsed")
    if isinstance(op, list) and op and isinstance(op[0], dict) and "label" in op[0]:
        obj = op[0]
        return str(obj["label"]).strip(), obj

    # Fallback: peel text from "message" output (not reasoning)
    text = ""
    try:
        for item in (resp_json.get("output") or []):
            if item.get("type") == "message" and item.get("content"):
                text = item["content"][0].get("text", "")
                if text:
                    break
        if not text and resp_json.get("output"):
            text = resp_json["output"][0]["content"][0].get("text", "")
    except Exception:
        pass

    try:
        obj = json.loads(text)
        if isinstance(obj, dict) and "label" in obj:
            return str(obj["label"]).strip(), obj
    except Exception:
        pass

    lab = _normalize_label(text, labels)
    return lab, ({"label": lab} if lab else {})

def _user_prompt(tweet_text: str, labels: List[str], rules: str) -> str:
    return (
        f"Classify the tweet into exactly one of: {', '.join(labels)}.\n"
        f"{rules or 'Return a JSON object that follows the provided schema.'}\n"
        f"Tweet: {tweet_text}"
    )

# ----------------------------- core probe ---------------------------------

def probe_classify(
    model: str,
    rows: pd.DataFrame,
    labels: List[str],
    rules: str = "",
    dump_first_raw: bool = True,
) -> pd.DataFrame:
    """
    rows: DataFrame with columns tweet_id, tweet_text, class_label (class_label optional for inference-only).
    returns: DataFrame with predicted_label (+confidence if present).
    """
    is_gpt5 = model.lower().startswith(("gpt-5", "o4", "o3"))  # Responses API family
    # Keep schema minimal for gpt-5-mini (omit confidence); strict SO for both paths
    schema = _schema(labels, keep_conf=not is_gpt5)

    out_rows = []
    dumped = False

    for _, r in rows.iterrows():
        user = _user_prompt(str(r["tweet_text"]), labels, rules)

        if is_gpt5:
            # ------- Responses API (Structured Outputs via text.format) -------
            body = {
                "model": model,
                "input": [
                    {"role": "system", "content": "Follow the schema exactly."},
                    {"role": "user",   "content": user},
                ],
                "text": {
                    "format": {
                        "type": "json_schema",
                        "name": "tweet_label",
                        "schema": schema,
                        "strict": True
                    }
                },
                # allow room for model's internal reasoning + final JSON
                "max_output_tokens": 300,
            }
            resp = requests.post(f"{OPENAI_BASE}/responses", headers=H_JSON, json=body, timeout=60)
        else:
            # ------- Chat Completions (4o / 4.1 / 4o-mini) -------
            body = {
                "model": model,
                "messages": [
                    {"role": "system", "content": "Follow the schema exactly."},
                    {"role": "user",   "content": user},
                ],
                "response_format": {
                    "type": "json_schema",
                    "json_schema": {"name": "tweet_label", "schema": schema, "strict": True}
                },
                "max_tokens": 40,
                "temperature": 0.0,
                "top_p": 1,
            }
            resp = requests.post(f"{OPENAI_BASE}/chat/completions", headers=H_JSON, json=body, timeout=60)

        if resp.status_code != 200:
            print(">>> API error body:", resp.text[:800])
            resp.raise_for_status()

        j = resp.json()

        # Some Responses API runs can be "incomplete"; poll a few times
        if is_gpt5 and j.get("status") == "incomplete":
            rid = j.get("id")
            import time
            for _ in range(10):
                time.sleep(1.8)
                poll = requests.get(f"{OPENAI_BASE}/responses/{rid}", headers={"Authorization": f"Bearer {API_KEY}"}, timeout=30)
                if poll.status_code == 200 and poll.json().get("status") in ("completed", "failed"):
                    j = poll.json()
                    break

        if dump_first_raw and not dumped:
            print("\n--- RAW first response ---")
            print(json.dumps(j, indent=2)[:1200])
            print("----------------------------------")
            dumped = True

        if is_gpt5:
            label, parsed = _extract_from_responses(j, labels)
        else:
            label, parsed = _extract_from_chat(j, labels)

        out_rows.append({
            "tweet_id": str(r.get("tweet_id", "")),
            "tweet_text": str(r.get("tweet_text", "")),
            "class_label": str(r.get("class_label", "")),
            "predicted_label": (label or ""),
            "confidence": (parsed.get("confidence") if isinstance(parsed, dict) else None),
        })

    out = pd.DataFrame(out_rows)
    # quick macro-F1 if truth exists
    if "class_label" in out.columns and out["class_label"].astype(str).str.len().gt(0).any():
        try:
            from sklearn.metrics import f1_score
            y_true = out["class_label"].tolist()
            y_pred = out["predicted_label"].tolist()
            print("Macro-F1:", f1_score(y_true, y_pred, average="macro"))
        except Exception:
            pass
    return out

# ----------------------------- quick demo ---------------------------------

def demo(model="gpt-5-mini"):
    df = pd.DataFrame([
        {"tweet_id": "t1", "tweet_text": "We need water and food urgently.",        "class_label": "Request-Help"},
        {"tweet_id": "t2", "tweet_text": "Roads are flooded near the river.",       "class_label": "Report-Event"},
        {"tweet_id": "t3", "tweet_text": "Thank you to the volunteers. Stay safe!", "class_label": "Other"},
    ])
    labels = ["Request-Help", "Report-Event", "Other"]
    res = probe_classify(model=model, rows=df, labels=labels, rules="Return strict JSON.")
    print(res)

if __name__ == "__main__":
    demo(os.getenv("TEST_MODEL", "gpt-5-mini"))



--- RAW first response ---
{
  "id": "resp_02aa79dffb3ea95f00690d7fe256c88193ab0eab529de1070d",
  "object": "response",
  "created_at": 1762492386,
  "status": "completed",
  "background": false,
  "billing": {
    "payer": "developer"
  },
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "max_output_tokens": 300,
  "max_tool_calls": null,
  "model": "gpt-5-mini-2025-08-07",
  "output": [
    {
      "id": "rs_02aa79dffb3ea95f00690d7fe328188193a6726c684014db14",
      "type": "reasoning",
      "summary": []
    },
    {
      "id": "msg_02aa79dffb3ea95f00690d7fe4061c819393b2ee6b3df5abb1",
      "type": "message",
      "status": "completed",
      "content": [
        {
          "type": "output_text",
          "annotations": [],
          "logprobs": [],
          "text": "{\"label\":\"Request-Help\"}"
        }
      ],
      "role": "assistant"
    }
  ],
  "parallel_tool_calls": true,
  "previous_response_id": null,
  "prompt_cache_key": null,
  "prompt_c

In [33]:
import pandas as pd
csv_predictions_path ="results/kerala_floods_2018/test/gpt-5-mini/20251107-171449-modeR-gpt-5-mini-RULES1-probe/predictions.csv"
df = pd.read_csv(csv_predictions_path)
df.head()
na_rows = df[df['predicted_label'].isna()]
na_rows


,tweet_id,tweet_text,class_label,predicted_label,confidence
115,1030902729109987328,#KeralaReliefFund I did my partὄDἿBὄDἿB I requ...,rescue_volunteering_or_donation_effort,NaN,NaN
150,1032543908780105733,RT @MoviePlanet8: #KeralaFloods Rescue Of A 2 ...,rescue_volunteering_or_donation_effort,NaN,NaN
154,1030746225631408128,"I have lived for 1 yr in Kerala, amazing place...",sympathy_and_support,NaN,NaN
301,1030416262394867714,#keralarain #keralaflood #keralatrafficupdates...,rescue_volunteering_or_donation_effort,NaN,NaN
320,1031204288125648896,Besides that PM has announced 2 Lakh pp to the...,infrastructure_and_utility_damage,NaN,NaN
357,1033129500580577281,#KeralaFloods : Hindu Mahasabhas Swami Chakrap...,rescue_volunteering_or_donation_effort,NaN,NaN
566,1030396917836967936,Stars indicate that there may be no immediate ...,sympathy_and_support,NaN,NaN
742,1035754762946469893,@Paytmcare @Paytm Can you guys start taking do...,rescue_volunteering_or_donation_effort,NaN,NaN
1319,1032822095506493440,Death toll nears 400 in Indias flood-hit Keral...,injured_or_dead_people,NaN,NaN
